# PEARLS AQI Predictor - SHAP Explainability Analysis

SHAP-based explainability analysis of the production 24-hour, 48-hour and 72-hour AQI forecasting models.

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
import shap
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().resolve().parent

DATA_FILE = BASE_DIR / "data" / "processed" / "ml_dataset.csv"
MODEL_DIR = BASE_DIR / "models" / "production"
OUTPUT_DIR = BASE_DIR / "data" / "processed" / "model_analysis" / "shap"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("PEARLS AQI PREDICTOR")
print("SHAP MODEL EXPLAINABILITY ANALYSIS")
print("=" * 60)

# Load data
df = pd.read_csv(DATA_FILE, parse_dates=["timestamp"])

# Load production feature list
with open(MODEL_DIR / "feature_list.json", "r", encoding="utf-8") as f:
    features = json.load(f)

print(f"Dataset shape: {df.shape}")
print(f"Production features: {len(features)}")

X = df[features].dropna().copy()

print(f"SHAP input shape: {X.shape}")

# Analyse all three production models
for horizon in ["24h", "48h", "72h"]:

    print("\n" + "=" * 60)
    print(f"SHAP ANALYSIS: {horizon}")
    print("=" * 60)

    model_path = MODEL_DIR / f"aqi_model_{horizon}.joblib"

    model = joblib.load(model_path)

    print("Model loaded.")

    # Use a representative sample to keep SHAP computation manageable
    sample_size = min(500, len(X))
    X_sample = X.tail(sample_size)

    print(f"Samples analysed: {len(X_sample)}")

    explainer = shap.TreeExplainer(model)

    shap_values = explainer.shap_values(X_sample)

    # Mean absolute SHAP importance
    importance = pd.DataFrame({
        "feature": features,
        "mean_abs_shap": abs(shap_values).mean(axis=0)
    })

    importance = importance.sort_values(
        "mean_abs_shap",
        ascending=False
    )

    output_csv = OUTPUT_DIR / f"shap_importance_{horizon}.csv"

    importance.to_csv(
        output_csv,
        index=False
    )

    print("\nTop 15 SHAP features:")
    print(importance.head(15).to_string(index=False))

    print(f"\nSaved: {output_csv}")

    # SHAP bar plot
    plt.figure()

    shap.summary_plot(
        shap_values,
        X_sample,
        plot_type="bar",
        show=False,
        max_display=20
    )

    plt.title(
        f"SHAP Feature Importance - {horizon} AQI Forecast"
    )

    plt.tight_layout()

    plot_file = OUTPUT_DIR / f"shap_summary_{horizon}.png"

    plt.savefig(
        plot_file,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {plot_file}")

print("\n" + "=" * 60)
print("SHAP ANALYSIS COMPLETED")
print("=" * 60)





C:\Users\Abubakar\Desktop\pearls-aqi-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PEARLS AQI PREDICTOR
SHAP MODEL EXPLAINABILITY ANALYSIS
Dataset shape: (2255, 77)
Production features: 25
SHAP input shape: (2255, 25)

SHAP ANALYSIS: 24h
Model loaded.
Samples analysed: 500



Top 15 SHAP features:
             feature  mean_abs_shap
                  o3       5.230297
          target_aqi       3.919894
                 day       3.542457
             dow_sin       3.108235
          feels_like       2.654798
          aqi_lag_1h       2.592775
         day_of_week       2.176907
aqi_rolling_mean_48h       1.971402
aqi_rolling_mean_24h       1.886000
         aqi_lag_48h       1.659163
 aqi_rolling_std_48h       1.070532
 aqi_rolling_std_24h       0.857167
                  co       0.850748
       aqi_change_3h       0.838033
pm25_rolling_mean_3h       0.785861

Saved: C:\Users\Abubakar\Desktop\pearls-aqi-predictor\data\processed\model_analysis\shap\shap_importance_24h.csv


Saved: C:\Users\Abubakar\Desktop\pearls-aqi-predictor\data\processed\model_analysis\shap\shap_summary_24h.png

SHAP ANALYSIS: 48h
Model loaded.
Samples analysed: 500



Top 15 SHAP features:
             feature  mean_abs_shap
                  o3       5.210579
                 day       4.205987
         day_of_week       3.578118
          feels_like       2.704944
             dow_sin       2.225836
          target_aqi       2.041212
 aqi_rolling_std_24h       2.024151
          aqi_lag_1h       1.458322
aqi_rolling_mean_24h       1.342975
aqi_rolling_mean_48h       1.328432
         aqi_lag_48h       0.924784
                  co       0.890530
 aqi_rolling_std_48h       0.842486
 aqi_rolling_std_12h       0.800111
pm25_rolling_mean_3h       0.720026

Saved: C:\Users\Abubakar\Desktop\pearls-aqi-predictor\data\processed\model_analysis\shap\shap_importance_48h.csv


Saved: C:\Users\Abubakar\Desktop\pearls-aqi-predictor\data\processed\model_analysis\shap\shap_summary_48h.png

SHAP ANALYSIS: 72h
Model loaded.
Samples analysed: 500



Top 15 SHAP features:
             feature  mean_abs_shap
                 day       6.129640
                  o3       5.955408
         day_of_week       2.941493
             dow_sin       2.245758
          target_aqi       1.889396
                  co       1.521992
          aqi_lag_1h       1.281193
 aqi_rolling_std_24h       1.265937
          feels_like       1.166044
aqi_rolling_mean_48h       1.111996
         aqi_lag_48h       1.039110
 aqi_rolling_std_48h       0.889895
 aqi_rolling_std_12h       0.792567
aqi_rolling_mean_24h       0.769543
pm25_rolling_mean_3h       0.719549

Saved: C:\Users\Abubakar\Desktop\pearls-aqi-predictor\data\processed\model_analysis\shap\shap_importance_72h.csv


Saved: C:\Users\Abubakar\Desktop\pearls-aqi-predictor\data\processed\model_analysis\shap\shap_summary_72h.png

SHAP ANALYSIS COMPLETED
